# Uber Pickups: hot zones in New York City
**Goal:** recommend to drivers the zones with the most pickup demand, for each day of week and hour, so that riders wait less than 5-7 minutes.

**Data:** Uber pickups in NYC, April to September 2014 (about 4.5 million pickups: date/time, latitude, longitude, base).

**Approach:**
1. Explore demand in time (month, day of week, hour) and space.
2. *Start small*: one slice (Monday, 17:00-18:00), tune **KMeans** and **DBSCAN**, compare them with silhouette, Davies-Bouldin and coverage (bonus: HDBSCAN).
3. *Grow big*: run both algorithms on the 168 slices (7 days x 24 hours), compare them with the business metric (coverage) and the share of noise.
4. Describe the hot zones per day of week and map them (plotly). Results also feed the Streamlit dashboard (`dashboard/`).

### Setup

In [1]:
import sys, subprocess, importlib
REQUIRED = {'pandas': 'pandas', 'numpy': 'numpy', 'sklearn': 'scikit-learn', 'plotly': 'plotly'}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
importlib.invalidate_caches()
print('Environment ready. Python:', sys.executable)

In [2]:
import os, time, warnings
import numpy as np, pandas as pd
import plotly.express as px, plotly.io as pio
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
warnings.filterwarnings('ignore', category=FutureWarning)
# numexpr returns empty selections on this large table in Anaconda base, which breaks the slices
pd.set_option("compute.use_numexpr", False)
# KMeans warns about a Windows memory leak on every slice, and its message prints a local path
warnings.filterwarnings("ignore", message=".*KMeans is known to have a memory leak.*")
pio.renderers.default = 'plotly_mimetype+notebook_connected'  # works in JupyterLab, VS Code and exported HTML
SEED = 42
DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
MAP_STYLE = 'carto-positron'  # free basemap, no token needed

## 1. Data
The Jedha zip (`uber-trip-data.zip`) is unzipped in `../data/uber-trip-data/`. The six monthly files `uber-raw-data-<month>14.csv` (April to September 2014) are read from there.

In [3]:
MONTHS = ['apr', 'aug', 'jul', 'jun', 'may', 'sep']
sources = [f'../data/uber-trip-data/uber-raw-data-{m}14.csv' for m in MONTHS]
raw = pd.concat([pd.read_csv(f) for f in sources], ignore_index=True)
raw.columns = ['datetime', 'lat', 'lon', 'base']
print(f'{len(raw):,} pickups from {len(sources)} files')
raw.head()

4,534,327 pickups from 6 files


,datetime,lat,lon,base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


In [4]:
raw['datetime'] = pd.to_datetime(raw['datetime'], format='%m/%d/%Y %H:%M:%S')
print('Missing values:', raw.isna().sum().sum(), '| period:', raw['datetime'].min(), '->', raw['datetime'].max())
raw[['lat', 'lon']].describe().round(4)

Missing values: 0 | period: 2014-04-01 00:00:00 -> 2014-09-30 22:59:00


,lat,lon
count,4.534327e+06,4.534327e+06
mean,4.073930e+01,-7.397300e+01
std,3.990000e-02,5.730000e-02
min,3.965690e+01,-7.492900e+01
25%,4.072110e+01,-7.399650e+01
50%,4.074220e+01,-7.398340e+01
75%,4.076100e+01,-7.396530e+01
max,4.211660e+01,-7.206660e+01


Coordinates go far outside the city (up to 42.1°N / -72.1°E). We keep the NYC metropolitan box (lat 40.5-41.0, lon -74.3 to -73.6), which still includes the three airports (JFK, LaGuardia, Newark).

For clustering, coordinates are projected to **kilometres** (equirectangular projection around NYC). In raw degrees, 1° of longitude is ~84 km and 1° of latitude ~111 km, so Euclidean distances on degrees would be distorted.

In [5]:
LAT0 = np.radians(40.75)
df = raw[raw['lat'].between(40.5, 41.0) & raw['lon'].between(-74.3, -73.6)].copy()
print(f'Removed outside the NYC box: {len(raw) - len(df):,} ({(len(raw) - len(df)) / len(raw):.2%})')
df['x_km'] = df['lon'] * np.cos(LAT0) * 111.32
df['y_km'] = df['lat'] * 110.57
df['month'] = df['datetime'].dt.month_name()
df['weekday'] = df['datetime'].dt.dayofweek
df['day_name'] = pd.Categorical(df['weekday'].map(dict(enumerate(DAYS))), categories=DAYS, ordered=True)
df['hour'] = df['datetime'].dt.hour
df['date'] = df['datetime'].dt.date
del raw

Removed outside the NYC box: 17,771 (0.39%)


## 2. When is the demand?

In [6]:
fig = px.bar(df.groupby('month', sort=False).size().reset_index(name='pickups'), x='month', y='pickups', title='Pickups per month (2014)',
             category_orders={'month': ['April', 'May', 'June', 'July', 'August', 'September']})
fig.show()

In [7]:
n_days = df.groupby('weekday')['date'].nunique()
by_day = (df.groupby('weekday').size() / n_days).rename('avg_pickups_per_day').reset_index()
by_day['day'] = by_day['weekday'].map(dict(enumerate(DAYS)))
px.bar(by_day, x='day', y='avg_pickups_per_day', title='Average pickups per day of week').show()

In [8]:
heat = df.groupby(['weekday', 'hour']).size().unstack() / n_days.values[:, None]
fig = px.imshow(heat.values, x=list(range(24)), y=DAYS, aspect='auto', color_continuous_scale='Viridis',
                labels=dict(x='hour', y='', color='avg pickups / hour'), title='Average pickups per hour, by day of week')
fig.show()
peak = heat.stack().idxmax(); low = heat.stack().idxmin()
print(f'Peak: {DAYS[peak[0]]} {peak[1]}:00 ({heat.max().max():.0f} pickups/hour on average); lowest: {DAYS[low[0]]} {low[1]}:00 ({heat.min().min():.0f})')

Peak: Thursday 17:00 (2174 pickups/hour on average); lowest: Tuesday 2:00 (95)


Demand grows over the 6 months, is highest on Thursday and Friday, then Wednesday and Saturday, and follows a commuter profile on weekdays (morning peak 7-8h, evening peak 17-18h) and a night-life profile on Friday/Saturday nights. A single static set of hot zones is therefore not enough: zones must be computed per day **and** hour.

## 3. Where is the demand?

In [9]:
sample = df.sample(60_000, random_state=SEED)
fig = px.density_map(sample, lat='lat', lon='lon', radius=4, zoom=10, center=dict(lat=40.73, lon=-73.96),
                     map_style=MAP_STYLE, title='Pickup density (random sample of 60,000 pickups)', height=600)
fig.show()

Demand is concentrated in Manhattan below Central Park, with secondary spots in Brooklyn (Williamsburg, Downtown Brooklyn), Queens (Long Island City, Astoria) and the airports.

To name the zones found by the algorithms, each zone center is matched with the nearest landmark of the list below (within 1.5 km).

In [10]:
LANDMARKS = {
    'Times Square / Midtown West': (40.7580, -73.9855), 'Grand Central / Midtown East': (40.7527, -73.9772),
    'Penn Station': (40.7506, -73.9935), 'Columbus Circle': (40.7681, -73.9819), 'Murray Hill / Gramercy': (40.7448, -73.9787),
    'Chelsea / Meatpacking': (40.7420, -74.0048), 'Union Square': (40.7359, -73.9911), 'West Village': (40.7358, -74.0036),
    'East Village': (40.7265, -73.9815), 'SoHo': (40.7233, -74.0030), 'Lower East Side': (40.7150, -73.9843),
    'Tribeca': (40.7163, -74.0086), 'Financial District': (40.7075, -74.0113), 'Upper East Side': (40.7736, -73.9566),
    'Upper West Side': (40.7870, -73.9754), 'Harlem': (40.8116, -73.9465), 'Williamsburg': (40.7081, -73.9571),
    'Downtown Brooklyn': (40.6928, -73.9903), 'Park Slope': (40.6710, -73.9814), 'Long Island City': (40.7447, -73.9485),
    'Astoria': (40.7644, -73.9235), 'JFK Airport': (40.6413, -73.7781), 'LaGuardia Airport': (40.7769, -73.8740),
    'Newark Airport': (40.6895, -74.1745),
}
LM = pd.DataFrame([(k, *v) for k, v in LANDMARKS.items()], columns=['name', 'lat', 'lon'])
LM['x_km'] = LM['lon'] * np.cos(LAT0) * 111.32; LM['y_km'] = LM['lat'] * 110.57

def name_zone(x, y, max_km=1.5):
    d = np.hypot(LM['x_km'] - x, LM['y_km'] - y)
    i = d.idxmin()
    return LM.loc[i, 'name'] if d[i] <= max_km else 'Other'

## 4. Start small: Monday, 17:00-18:00
All Monday pickups between 17:00 and 18:00 over the 6 months. A random sample of at most 10,000 points is used for each slice (enough for stable centers, fast enough to repeat on 168 slices).

In [11]:
MAX_POINTS = 10_000
ref = df[(df['weekday'] == 0) & (df['hour'] == 17)]
print(f'{len(ref):,} pickups in the slice')
ref = ref.sample(min(len(ref), MAX_POINTS), random_state=SEED)
X = ref[['x_km', 'y_km']].values

41,905 pickups in the slice


### 4.1 KMeans: choice of k (elbow + silhouette)

In [12]:
rows = []
for k in range(3, 21):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    rows.append({'k': k, 'inertia': km.inertia_, 'silhouette': silhouette_score(X, km.labels_, sample_size=5000, random_state=SEED)})
k_scores = pd.DataFrame(rows)
fig = px.line(k_scores, x='k', y=['inertia'], markers=True, title='Elbow: inertia vs k'); fig.show()
fig = px.line(k_scores, x='k', y='silhouette', markers=True, title='Silhouette vs k'); fig.show()
k_scores.round(3).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
k,3.000,4.000,5.000,6.000,7.000,8.000,9.000,10.000,11.000,12.000,13.000,14.000,15.000,16.000,17.000,18.000,19.000,20.000
inertia,121133.213,91552.705,71464.449,55520.892,45654.893,38049.468,33480.789,29006.100,26237.263,23998.600,22235.768,20335.982,18832.493,17732.441,16698.901,15700.622,14896.413,14066.346
silhouette,0.446,0.495,0.506,0.480,0.475,0.411,0.425,0.434,0.437,0.439,0.407,0.408,0.409,0.412,0.409,0.412,0.393,0.382


The silhouette is highest for small k, with its maximum at k = 5: Manhattan vs the rest. But 4 zones covering the whole city are useless for a driver. The elbow flattens around **k = 10**, where the silhouette stays at about the same level as for k = 8-12: k = 10 gives neighbourhood-sized zones. This business constraint (zones of a few km) drives the choice more than the silhouette maximum.

In [13]:
K = 10
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X)
ref['kmeans'] = km.labels_

def zones_table(points, labels):
    t = (points.assign(cluster=labels).query('cluster >= 0').groupby('cluster')
         .agg(lat=('lat', 'mean'), lon=('lon', 'mean'), x_km=('x_km', 'mean'), y_km=('y_km', 'mean'), pickups=('lat', 'size')))
    t['share'] = t['pickups'] / len(points)
    t['zone'] = [name_zone(x, y) for x, y in zip(t['x_km'], t['y_km'])]
    return t.sort_values('pickups', ascending=False).reset_index()

def zone_map(points, labels, zones, title):
    p = points.assign(cluster=np.where(labels >= 0, labels.astype(str), 'noise'))
    fig = px.scatter_map(p, lat='lat', lon='lon', color='cluster', opacity=0.35, zoom=10.3, height=600,
                         center=dict(lat=40.74, lon=-73.97), map_style=MAP_STYLE, title=title,
                         color_discrete_map={'noise': 'lightgrey'})
    fig.update_traces(marker=dict(size=4))
    c = px.scatter_map(zones, lat='lat', lon='lon', size='pickups', hover_name='zone', size_max=30)
    c.update_traces(marker=dict(color='black'))
    fig.add_traces(c.data)
    fig.update_layout(showlegend=False)
    return fig

km_zones = zones_table(ref, km.labels_)
zone_map(ref, km.labels_, km_zones, 'KMeans (k=10), Monday 17:00-18:00: black = zone centers, size = pickups').show()
km_zones[['zone', 'lat', 'lon', 'pickups', 'share']].round(4)

,zone,lat,lon,pickups,share
0,Grand Central / Midtown East,40.7592,-73.9768,3005,0.3005
1,Penn Station,40.7450,-73.9934,2495,0.2495
2,SoHo,40.7191,-74.0023,2038,0.2038
3,Upper East Side,40.7824,-73.9602,897,0.0897
4,Park Slope,40.6704,-73.9781,448,0.0448
5,Williamsburg,40.7119,-73.9446,375,0.0375
6,Other,40.7642,-73.8645,306,0.0306
7,JFK Airport,40.6529,-73.7800,258,0.0258
8,Other,40.8615,-73.9141,97,0.0097
9,Newark Airport,40.6917,-74.1800,81,0.0081


### 4.2 DBSCAN: choice of eps and min_samples
DBSCAN works on the km coordinates: `eps` is a radius in km. `min_samples` is expressed as a share of the sample (0.5% of the points): 50 points when the sample has 10,000 points, fewer for the quietest slices.

In [14]:
rows = []
for eps in [0.10, 0.15, 0.20, 0.25, 0.30]:
    for frac in [0.003, 0.005, 0.01]:
        ms = max(10, int(frac * len(X)))
        lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
        sizes = pd.Series(lab[lab >= 0]).value_counts()
        rows.append({'eps_km': eps, 'min_samples_frac': frac, 'min_samples': ms, 'n_zones': len(sizes),
                     'noise_share': (lab == -1).mean(), 'largest_zone_share': sizes.max() / len(X) if len(sizes) else np.nan})
db_grid = pd.DataFrame(rows)
db_grid.round(3)

,eps_km,min_samples_frac,min_samples,n_zones,noise_share,largest_zone_share
0,0.10,0.003,30,34,0.622,0.128
1,0.10,0.005,50,11,0.874,0.032
2,0.10,0.010,100,1,0.990,0.010
3,0.15,0.003,30,23,0.376,0.377
4,0.15,0.005,50,14,0.544,0.249
5,0.15,0.010,100,3,0.907,0.045
6,0.20,0.003,30,10,0.267,0.647
7,0.20,0.005,50,13,0.365,0.407
8,0.20,0.010,100,10,0.616,0.236
9,0.25,0.003,30,12,0.204,0.742


Trade-off: with a large `eps` the whole of Midtown/Downtown Manhattan merges into one giant zone (no recommendation value); with a small `eps` or a high `min_samples` most points become noise. **eps = 0.2 km, min_samples = 0.5% of the slice** keeps about a dozen dense zones with a third of points as noise. DBSCAN zones are the *densest cores*, not a partition of the city.

In [15]:
EPS_KM, MIN_FRAC = 0.20, 0.005
db_labels = DBSCAN(eps=EPS_KM, min_samples=max(10, int(MIN_FRAC * len(X)))).fit_predict(X)
ref['dbscan'] = db_labels
db_zones = zones_table(ref, db_labels)
zone_map(ref, db_labels, db_zones, 'DBSCAN (eps=200 m, min_samples=0.5%), Monday 17:00-18:00 (grey = noise)').show()
db_zones[['zone', 'lat', 'lon', 'pickups', 'share']].round(4)

,zone,lat,lon,pickups,share
0,Grand Central / Midtown East,40.7530,-73.9821,4073,0.4073
1,SoHo,40.7237,-74.0017,946,0.0946
2,Financial District,40.7067,-74.0083,215,0.0215
3,Chelsea / Meatpacking,40.7419,-74.0066,196,0.0196
4,Tribeca,40.7151,-74.0133,178,0.0178
5,Penn Station,40.7566,-74.0003,171,0.0171
6,Chelsea / Meatpacking,40.7501,-74.0055,143,0.0143
7,JFK Airport,40.6449,-73.7821,109,0.0109
8,Upper East Side,40.7752,-73.9624,105,0.0105
9,LaGuardia Airport,40.7740,-73.8724,87,0.0087


### 4.3 Bonus: HDBSCAN
Hierarchical DBSCAN adapts to varying densities (no single `eps`). `leaf` selection returns the finest stable clusters.

In [16]:
hdb_labels = HDBSCAN(min_cluster_size=100, min_samples=30, cluster_selection_method='leaf', copy=True).fit_predict(X)
hdb_zones = zones_table(ref, hdb_labels)
zone_map(ref, hdb_labels, hdb_zones, 'HDBSCAN (leaf, min_cluster_size=100), Monday 17:00-18:00').show()
print(f'HDBSCAN: {len(hdb_zones)} zones, noise {np.mean(hdb_labels == -1):.0%}')

HDBSCAN: 17 zones, noise 62%


### 4.4 Comparison on the reference slice
- **Silhouette** and **Davies-Bouldin** (computed on clustered points only, noise excluded for DBSCAN / HDBSCAN): cluster separation. Higher silhouette / lower DB is better.
- **Coverage within 1 km**: share of *all* sampled pickups of the slice, noise included, that are less than 1 km from a zone center. In Manhattan traffic 1 km is roughly a 4-6 minute drive (an assumption), close to the 5-7 minutes of the brief. This is the business metric.

In [17]:
COVER_KM = 1.0

def coverage(points, zones, radius=COVER_KM):
    if len(zones) == 0:
        return 0.0
    d = np.hypot(points['x_km'].values[:, None] - zones['x_km'].values[None, :],
                 points['y_km'].values[:, None] - zones['y_km'].values[None, :]).min(axis=1)
    return float((d <= radius).mean())

def evaluate(name, labels, zones):
    mask = labels >= 0
    ok = mask.sum() > 1 and len(np.unique(labels[mask])) > 1
    return {'algorithm': name, 'n_zones': len(zones), 'noise_share': float(np.mean(labels == -1)),
            'silhouette': silhouette_score(X[mask], labels[mask], sample_size=5000, random_state=SEED) if ok else np.nan,
            'davies_bouldin': davies_bouldin_score(X[mask], labels[mask]) if ok else np.nan,
            'coverage_1km': coverage(ref, zones), 'top_zone_share': zones['share'].max()}

ref_compare = pd.DataFrame([evaluate('KMeans (k=10)', km.labels_, km_zones), evaluate('DBSCAN (eps=200 m)', db_labels, db_zones),
                            evaluate('HDBSCAN (leaf)', hdb_labels, hdb_zones)]).set_index('algorithm')
ref_compare.round(3)

,n_zones,noise_share,silhouette,davies_bouldin,coverage_1km,top_zone_share
algorithm,,,,,,
KMeans (k=10),10,0.000,0.434,0.666,0.522,0.300
DBSCAN (eps=200 m),13,0.365,0.165,0.543,0.558,0.407
HDBSCAN (leaf),17,0.616,0.595,0.472,0.764,0.078


## 5. Grow big: hot zones for every day of week and hour
Both algorithms are run with the parameters chosen above, except `n_init` (4 instead of 10), on each of the 168 slices (7 days x 24 hours, sample of at most 10,000 pickups per slice; 32 slices have fewer).

In [18]:
def hot_zones(points, method):
    Xs = points[['x_km', 'y_km']].values
    if method == 'kmeans':
        labels = KMeans(n_clusters=K, n_init=4, random_state=SEED).fit_predict(Xs)
    else:
        labels = DBSCAN(eps=EPS_KM, min_samples=max(10, int(MIN_FRAC * len(Xs)))).fit_predict(Xs)
    return labels, zones_table(points, labels)

t0 = time.time()
zone_rows, metric_rows = [], []
for wd in range(7):
    for h in range(24):
        s = df[(df['weekday'] == wd) & (df['hour'] == h)]
        n_slice = len(s)
        s = s.sample(min(len(s), MAX_POINTS), random_state=SEED)
        for method in ['kmeans', 'dbscan']:
            labels, z = hot_zones(s, method)
            z = z.assign(weekday=wd, day=DAYS[wd], hour=h, method=method, rank=np.arange(1, len(z) + 1),
                         avg_pickups_per_hour=z['share'] * n_slice / n_days[wd])
            zone_rows.append(z)
            metric_rows.append({'weekday': wd, 'day': DAYS[wd], 'hour': h, 'method': method, 'slice_pickups': n_slice,
                                'n_zones': len(z), 'noise_share': float(np.mean(labels == -1)), 'coverage_1km': coverage(s, z)})
all_zones = pd.concat(zone_rows, ignore_index=True)
metrics = pd.DataFrame(metric_rows)
print(f'{len(metrics)} clusterings in {time.time() - t0:.0f} s, {len(all_zones):,} zones')

336 clusterings in 14 s, 4,324 zones


In [19]:
summary = metrics.groupby('method').agg(avg_zones=('n_zones', 'mean'), min_zones=('n_zones', 'min'), max_zones=('n_zones', 'max'),
                                          avg_noise=('noise_share', 'mean'), avg_coverage_1km=('coverage_1km', 'mean'),
                                          min_coverage_1km=('coverage_1km', 'min'))
summary.round(3)

,avg_zones,min_zones,max_zones,avg_noise,avg_coverage_1km,min_coverage_1km
method,,,,,,
dbscan,15.738,6,36,0.503,0.561,0.247
kmeans,10.000,10,10,0.000,0.389,0.191


DBSCAN covers more pickups within 1 km on average, but it also places more pins (and a variable number). For a fair comparison, KMeans is re-run on every slice with **the same number of zones as DBSCAN** found for that slice.

In [20]:
fair_rows = []
for (wd, h), m in metrics[metrics['method'] == 'dbscan'].set_index(['weekday', 'hour']).iterrows():
    s = df[(df['weekday'] == wd) & (df['hour'] == h)]
    s = s.sample(min(len(s), MAX_POINTS), random_state=SEED)
    k = int(max(m['n_zones'], 2))
    labels = KMeans(n_clusters=k, n_init=4, random_state=SEED).fit_predict(s[['x_km', 'y_km']].values)
    fair_rows.append({'weekday': wd, 'hour': h, 'k': k, 'coverage_1km': coverage(s, zones_table(s, labels))})
fair = pd.DataFrame(fair_rows)
fair_compare = pd.DataFrame({
    'KMeans k=10': [10, metrics.query("method == 'kmeans'")['coverage_1km'].mean()],
    'KMeans k = DBSCAN zones': [fair['k'].mean(), fair['coverage_1km'].mean()],
    'DBSCAN': [metrics.query("method == 'dbscan'")['n_zones'].mean(), metrics.query("method == 'dbscan'")['coverage_1km'].mean()],
}, index=['avg_zones', 'avg_coverage_1km']).T
fair_compare.round(3)

,avg_zones,avg_coverage_1km
KMeans k=10,10.000,0.389
KMeans k = DBSCAN zones,15.738,0.519
DBSCAN,15.738,0.561


In [21]:
fig = px.line(metrics.assign(slot=metrics['weekday'] * 24 + metrics['hour']), x='slot', y='coverage_1km', color='method',
              title='Share of pickups within 1 km of a hot zone, for each of the 168 slices (0 = Monday 0h)',
              hover_data=['day', 'hour', 'n_zones'])
fig.update_yaxes(tickformat='.0%'); fig.show()

## 6. Hot zones per day of week

In [22]:
top_by_day = (all_zones[all_zones['method'] == 'kmeans']
    .groupby(['day', 'zone'], observed=True)['avg_pickups_per_hour'].sum().reset_index()
    .sort_values(['day', 'avg_pickups_per_hour'], ascending=[True, False]))
top_by_day['day'] = pd.Categorical(top_by_day['day'], DAYS, ordered=True)
description = (top_by_day[top_by_day['zone'] != 'Other'].sort_values(['day', 'avg_pickups_per_hour'], ascending=[True, False])
               .groupby('day', observed=True).head(5).groupby('day', observed=True)['zone'].apply(lambda z: ', '.join(z)))
description.rename('Top 5 named hot zones (KMeans, weighted by pickups over the day)').to_frame()

,"Top 5 named hot zones (KMeans, weighted by pickups over the day)"
day,
Monday,"SoHo, Upper East Side, Times Square / Midtown ..."
Tuesday,"SoHo, Times Square / Midtown West, Upper East ..."
Wednesday,"Times Square / Midtown West, SoHo, Union Squar..."
Thursday,"SoHo, Times Square / Midtown West, Union Squar..."
Friday,"SoHo, Times Square / Midtown West, Union Squar..."
Saturday,"SoHo, Upper East Side, Times Square / Midtown ..."
Sunday,"SoHo, Times Square / Midtown West, Upper East ..."


In [23]:
def peak_zone(d):
    sub = all_zones[(all_zones['method'] == 'kmeans') & (all_zones['day'] == d)]
    best = sub.sort_values('avg_pickups_per_hour', ascending=False).iloc[0]
    return f"{best['zone']} at {int(best['hour'])}:00 (~{best['avg_pickups_per_hour']:.0f} pickups/hour in the zone)"

for d in DAYS:
    print(f'{d:<10} busiest zone-hour: {peak_zone(d)}')

Monday     busiest zone-hour: Grand Central / Midtown East at 17:00 (~473 pickups/hour in the zone)
Tuesday    busiest zone-hour: Grand Central / Midtown East at 17:00 (~693 pickups/hour in the zone)
Wednesday  busiest zone-hour: Times Square / Midtown West at 17:00 (~711 pickups/hour in the zone)
Thursday   busiest zone-hour: Grand Central / Midtown East at 18:00 (~680 pickups/hour in the zone)
Friday     busiest zone-hour: SoHo at 22:00 (~833 pickups/hour in the zone)
Saturday   busiest zone-hour: SoHo at 23:00 (~768 pickups/hour in the zone)
Sunday     busiest zone-hour: SoHo at 0:00 (~395 pickups/hour in the zone)


### Maps per day of week
Zone centers for each hour of the selected day; use the slider to move through the 168 day-hour steps. Circle size = average pickups per hour in the zone.

In [24]:
def day_hour_map(method, title):
    z = all_zones[all_zones['method'] == method].copy()
    z['frame'] = z['weekday'].astype(str) + ' ' + z['day'] + ' ' + z['hour'].astype(str).str.zfill(2) + 'h'
    z = z.sort_values(['weekday', 'hour'])
    fig = px.scatter_map(z, lat='lat', lon='lon', size='avg_pickups_per_hour', color='day', animation_frame='frame',
                         hover_name='zone', hover_data={'avg_pickups_per_hour': ':.0f', 'rank': True, 'frame': False},
                         size_max=35, zoom=10.3, center=dict(lat=40.74, lon=-73.96), height=650, map_style=MAP_STYLE,
                         category_orders={'day': DAYS}, title=title)
    return fig

fig_km = day_hour_map('kmeans', 'KMeans hot zones by day of week and hour (slider)')
fig_km.show()

In [25]:
fig_db = day_hour_map('dbscan', 'DBSCAN hot zones by day of week and hour (slider)')
fig_db.show()

In [26]:
# one static map per day of week: zones of the whole day (KMeans on all hours of that day)
day_rows = []
for wd in range(7):
    s = df[df['weekday'] == wd].sample(MAX_POINTS, random_state=SEED)
    _, z = hot_zones(s, 'kmeans')
    day_rows.append(z.assign(day=DAYS[wd]))
day_zones = pd.concat(day_rows)
fig = px.scatter_map(day_zones, lat='lat', lon='lon', size='share', color='day', hover_name='zone', size_max=25, zoom=10,
                     center=dict(lat=40.74, lon=-73.96), height=650, map_style=MAP_STYLE, category_orders={'day': DAYS},
                     title='KMeans hot zones of each day of week (all hours together), click the legend to filter')
fig.show()

Weekday rush hours vs weekend nights (KMeans zones, sum of average pickups per hour over the slices):

In [27]:
def zone_weights(days, hours):
    z = all_zones[(all_zones['method'] == 'kmeans') & all_zones['day'].isin(days) & all_zones['hour'].isin(hours)]
    return z.groupby('zone')['avg_pickups_per_hour'].sum().sort_values(ascending=False).head(8).round(0)

pd.concat({'Mon-Thu 7-8h & 17-18h': zone_weights(DAYS[:4], [7, 8, 17, 18]).reset_index(drop=False).rename(columns={'avg_pickups_per_hour': 'weight'}),
           'Fri-Sat 22h-2h': zone_weights(['Friday', 'Saturday'], [22, 23, 0, 1, 2]).reset_index(drop=False).rename(columns={'avg_pickups_per_hour': 'weight'})}, axis=1)

Mon-Thu 7-8h & 17-18h                       Fri-Sat 22h-2h        
                           zone  weight                         zone  weight
0                          SoHo  6015.0                         SoHo  4058.0
1  Grand Central / Midtown East  4298.0  Times Square / Midtown West  2041.0
2               Upper East Side  3744.0                 Williamsburg   985.0
3                  Union Square  3710.0        Chelsea / Meatpacking   877.0
4   Times Square / Midtown West  2950.0                   Park Slope   784.0
5                    Park Slope  1150.0                        Other   715.0
6                         Other  1072.0              Upper East Side   392.0
7                  Williamsburg   939.0              Columbus Circle   316.0

## 7. Exports

In [28]:
os.makedirs('../maps', exist_ok=True); os.makedirs('../dashboard', exist_ok=True)
fig_km.write_html('../maps/hotzones_kmeans_by_day_hour.html', include_plotlyjs=True)
fig_db.write_html('../maps/hotzones_dbscan_by_day_hour.html', include_plotlyjs=True)
fig.write_html('../maps/hotzones_by_day.html', include_plotlyjs=True)
cols = ['method', 'weekday', 'day', 'hour', 'rank', 'zone', 'lat', 'lon', 'pickups', 'share', 'avg_pickups_per_hour']
all_zones[cols].to_csv('../dashboard/hotzones.csv', index=False)
metrics.to_csv('../dashboard/metrics.csv', index=False)
heat.to_csv('../dashboard/demand_heatmap.csv')
df.sample(150_000, random_state=SEED)[['lat', 'lon', 'weekday', 'hour']].to_csv('../dashboard/pickups_sample.csv.gz', index=False)
print('Saved maps/*.html and dashboard/*.csv')

Saved maps/*.html and dashboard/*.csv


## 8. Conclusion

In [29]:
r = ref_compare; s_ = summary
print(f"Reference slice (Monday 17h): KMeans {r.loc['KMeans (k=10)', 'n_zones']} zones covering {r.loc['KMeans (k=10)', 'coverage_1km']:.0%} of pickups within 1 km; "
      f"DBSCAN {r.loc['DBSCAN (eps=200 m)', 'n_zones']} zones, {r.loc['DBSCAN (eps=200 m)', 'noise_share']:.0%} noise, coverage {r.loc['DBSCAN (eps=200 m)', 'coverage_1km']:.0%}.")
print(f"All 168 slices: KMeans average coverage {s_.loc['kmeans', 'avg_coverage_1km']:.0%} (min {s_.loc['kmeans', 'min_coverage_1km']:.0%}); "
      f"DBSCAN average coverage {s_.loc['dbscan', 'avg_coverage_1km']:.0%} with {s_.loc['dbscan', 'avg_zones']:.1f} zones on average "
      f"(between {s_.loc['dbscan', 'min_zones']} and {s_.loc['dbscan', 'max_zones']}).")

Reference slice (Monday 17h): KMeans 10 zones covering 52% of pickups within 1 km; DBSCAN 13 zones, 36% noise, coverage 56%.
All 168 slices: KMeans average coverage 39% (min 19%); DBSCAN average coverage 56% with 15.7 zones on average (between 6 and 36).


In [30]:
fc = fair_compare
print(f"Same number of pins: KMeans covers {fc.loc['KMeans k = DBSCAN zones', 'avg_coverage_1km']:.0%} of pickups within 1 km vs {fc.loc['DBSCAN', 'avg_coverage_1km']:.0%} for DBSCAN.")

Same number of pins: KMeans covers 52% of pickups within 1 km vs 56% for DBSCAN.


**Reading the comparison**
- **KMeans** returns a fixed number of zones that partition the whole demand, so the number of pins is predictable for the app. With k = 10 its centers are pulled toward the outer, sparser areas (Brooklyn, airports, Bronx), which lowers the share of pickups close to a pin.
- **DBSCAN** keeps only dense cores and labels the rest as noise: its pins sit in the busiest blocks and the number of zones adapts to each hour. Its limits: one `eps` for the whole city, a very large merged zone in Manhattan at busy hours, and sparse but real spots (airports at night) can disappear into noise.
- The fair comparison above (same number of pins) separates the effect of the algorithm from the effect of the number of pins.
- **HDBSCAN** (bonus, reference slice only) gave the best silhouette and coverage on Monday 17h but labels most points as noise and needs its own tuning per slice before being generalised.

**For the app:** zones must be recomputed per day of week and hour. The table above shows the shift: at weekday rush hours, Grand Central / Midtown East, the Upper East Side and Union Square come right after Lower Manhattan (SoHo), while on Friday and Saturday nights SoHo stays first, Times Square / Midtown West is second, and Williamsburg and Chelsea / Meatpacking climb to the top 4. The night filter takes hours 22 to 2 of the same calendar days: it includes the end of Thursday night and leaves out the end of Saturday night. The dashboard lets the team switch between both algorithms on any slice.

**Limits and next steps:** 2014 data (6 months), pickups only (no driver supply, no wait times): hot zones show demand, not the gap between demand and supply. Next steps: add supply data, weight recent weeks more, forecast next-hour demand per zone.